# Author : Prashanta Pal

# Model : Hate Speech Detection by LSTM

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset,DataLoader
import torch.optim as optim
from collections import Counter
from nltk import word_tokenize
import nltk

In [2]:
import kagglehub
import os

# Download latest version
path = kagglehub.dataset_download("mrmorj/hate-speech-and-offensive-language-dataset")

print("Path to dataset files:", path)

100%|██████████| 1.01M/1.01M [00:00<00:00, 92.0MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/mrmorj/hate-speech-and-offensive-language-dataset/versions/1


In [10]:
# Load CSV file
df = pd.read_csv(
    os.path.join(path, "labeled_data.csv")
)

# Show first rows
print(df.head())

   Unnamed: 0  count  hate_speech  offensive_language  neither  class  \
0           0      3            0                   0        3      2   
1           1      3            0                   3        0      1   
2           2      3            0                   3        0      1   
3           3      3            0                   2        1      1   
4           4      6            0                   6        0      1   

                                               tweet  
0  !!! RT @mayasolovely: As a woman you shouldn't...  
1  !!!!! RT @mleew17: boy dats cold...tyga dwn ba...  
2  !!!!!!! RT @UrKindOfBrand Dawg!!!! RT @80sbaby...  
3  !!!!!!!!! RT @C_G_Anderson: @viva_based she lo...  
4  !!!!!!!!!!!!! RT @ShenikaRoberts: The shit you...  


In [4]:
df.shape

(24783, 7)

In [5]:
sub_df=df.sample(n=600)

In [9]:
sub_df

,Unnamed: 0,count,hate_speech,offensive_language,neither,class,tweet
8864,9110,6,0,6,0,1,Drop it down for a niggah
395,400,3,1,0,2,2,"""@socass_: Dude remember this wop video? @00se..."
898,917,3,0,3,0,1,"#porn,#android,#iphone,#ipad,#sex,#xxx, | #Mas..."
20248,20693,3,0,3,0,1,RT @tupactopus: need a bad bitch who knows how...
14588,14937,3,0,3,0,1,RT @Cam_Major: naw no bs tho...if u under lets...
...,...,...,...,...,...,...,...
20970,21421,3,0,0,3,2,"So my light was off &amp; door was closed , mo..."
14199,14540,3,0,3,0,1,RT @Annnna_dav69: Ima make you my bitch &#128521;
6189,6366,3,1,2,0,1,@itsjustmexo ur a cunt nd a wish a was a faggot
18325,18732,3,0,0,3,2,RT @_MykalaJ: @whitakerc54 blame it on the yel...


In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [14]:
sub_df.drop(columns=["Unnamed: 0" , "count",  "hate_speech",  "offensive_language",  "neither"],inplace=True)

In [21]:
sub_df["cleaned_tweet"]=sub_df["tweet"].str.replace(r"[^a-zA-z]"," ",regex=True)

In [22]:
sub_df

,class,tweet,cleaned_tweet
8864,1,Drop it down for a niggah,Drop it down for a niggah
395,2,"""@socass_: Dude remember this wop video? @00se...",socass_ Dude remember this wop video se...
898,1,"#porn,#android,#iphone,#ipad,#sex,#xxx, | #Mas...",porn android iphone ipad sex xxx Mas...
20248,1,RT @tupactopus: need a bad bitch who knows how...,RT tupactopus need a bad bitch who knows how...
14588,1,RT @Cam_Major: naw no bs tho...if u under lets...,RT Cam_Major naw no bs tho if u under lets...
...,...,...,...
20970,2,"So my light was off &amp; door was closed , mo...",So my light was off amp door was closed mo...
14199,1,RT @Annnna_dav69: Ima make you my bitch &#128521;,RT Annnna_dav Ima make you my bitch
6189,1,@itsjustmexo ur a cunt nd a wish a was a faggot,itsjustmexo ur a cunt nd a wish a was a faggot
18325,2,RT @_MykalaJ: @whitakerc54 blame it on the yel...,RT _MykalaJ whitakerc blame it on the yel...


In [23]:
sub_df["cleaned_tweet"]=sub_df["cleaned_tweet"].str.replace(r"[\s]+"," ",regex=True)

In [24]:
sub_df

,class,tweet,cleaned_tweet
8864,1,Drop it down for a niggah,Drop it down for a niggah
395,2,"""@socass_: Dude remember this wop video? @00se...",socass_ Dude remember this wop video sexilexi...
898,1,"#porn,#android,#iphone,#ipad,#sex,#xxx, | #Mas...",porn android iphone ipad sex xxx Masturbation...
20248,1,RT @tupactopus: need a bad bitch who knows how...,RT tupactopus need a bad bitch who knows how t...
14588,1,RT @Cam_Major: naw no bs tho...if u under lets...,RT Cam_Major naw no bs tho if u under lets say...
...,...,...,...
20970,2,"So my light was off &amp; door was closed , mo...",So my light was off amp door was closed mom wa...
14199,1,RT @Annnna_dav69: Ima make you my bitch &#128521;,RT Annnna_dav Ima make you my bitch
6189,1,@itsjustmexo ur a cunt nd a wish a was a faggot,itsjustmexo ur a cunt nd a wish a was a faggot
18325,2,RT @_MykalaJ: @whitakerc54 blame it on the yel...,RT _MykalaJ whitakerc blame it on the yellow c...


In [27]:
sub_df.drop(columns=["tweet"],inplace=True)

In [28]:
sub_df

,class,cleaned_tweet
8864,1,Drop it down for a niggah
395,2,socass_ Dude remember this wop video sexilexi...
898,1,porn android iphone ipad sex xxx Masturbation...
20248,1,RT tupactopus need a bad bitch who knows how t...
14588,1,RT Cam_Major naw no bs tho if u under lets say...
...,...,...
20970,2,So my light was off amp door was closed mom wa...
14199,1,RT Annnna_dav Ima make you my bitch
6189,1,itsjustmexo ur a cunt nd a wish a was a faggot
18325,2,RT _MykalaJ whitakerc blame it on the yellow c...


# Lemmatization ( Converting into Base Words)

In [29]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

In [34]:
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [31]:
def lemmatization(text):
  tokens = word_tokenize(text)

  lemmatizer = WordNetLemmatizer()

  lemmatized_words = [lemmatizer.lemmatize(word) for word in tokens]

  return " ".join(lemmatized_words)

In [35]:
sub_df["lemma_tweets"]=sub_df["cleaned_tweet"].apply(lemmatization)

In [36]:
sub_df

,class,cleaned_tweet,lemma_tweets
8864,1,Drop it down for a niggah,Drop it down for a niggah
395,2,socass_ Dude remember this wop video sexilexi...,socass_ Dude remember this wop video sexilexi ...
898,1,porn android iphone ipad sex xxx Masturbation...,porn android iphone ipad sex xxx Masturbation ...
20248,1,RT tupactopus need a bad bitch who knows how t...,RT tupactopus need a bad bitch who know how tr...
14588,1,RT Cam_Major naw no bs tho if u under lets say...,RT Cam_Major naw no b tho if u under let say a...
...,...,...,...
20970,2,So my light was off amp door was closed mom wa...,So my light wa off amp door wa closed mom walk...
14199,1,RT Annnna_dav Ima make you my bitch,RT Annnna_dav Ima make you my bitch
6189,1,itsjustmexo ur a cunt nd a wish a was a faggot,itsjustmexo ur a cunt nd a wish a wa a faggot
18325,2,RT _MykalaJ whitakerc blame it on the yellow c...,RT _MykalaJ whitakerc blame it on the yellow c...


In [37]:
nltk.download('stopwords')
from nltk.corpus import stopwords

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [38]:
def stop_words(text):
  stop_words = set(stopwords.words('english'))
  tokens = word_tokenize(text)
  filtered = [
    word for word in tokens
    if word.lower() not in stop_words
  ]
  return " ".join(filtered)

In [39]:
sub_df["final_tweets"]=sub_df["lemma_tweets"].apply(lemmatization)

In [40]:
sub_df

,class,cleaned_tweet,lemma_tweets,final_tweets
8864,1,Drop it down for a niggah,Drop it down for a niggah,Drop it down for a niggah
395,2,socass_ Dude remember this wop video sexilexi...,socass_ Dude remember this wop video sexilexi ...,socass_ Dude remember this wop video sexilexi ...
898,1,porn android iphone ipad sex xxx Masturbation...,porn android iphone ipad sex xxx Masturbation ...,porn android iphone ipad sex xxx Masturbation ...
20248,1,RT tupactopus need a bad bitch who knows how t...,RT tupactopus need a bad bitch who know how tr...,RT tupactopus need a bad bitch who know how tr...
14588,1,RT Cam_Major naw no bs tho if u under lets say...,RT Cam_Major naw no b tho if u under let say a...,RT Cam_Major naw no b tho if u under let say a...
...,...,...,...,...
20970,2,So my light was off amp door was closed mom wa...,So my light wa off amp door wa closed mom walk...,So my light wa off amp door wa closed mom walk...
14199,1,RT Annnna_dav Ima make you my bitch,RT Annnna_dav Ima make you my bitch,RT Annnna_dav Ima make you my bitch
6189,1,itsjustmexo ur a cunt nd a wish a was a faggot,itsjustmexo ur a cunt nd a wish a wa a faggot,itsjustmexo ur a cunt nd a wish a wa a faggot
18325,2,RT _MykalaJ whitakerc blame it on the yellow c...,RT _MykalaJ whitakerc blame it on the yellow c...,RT _MykalaJ whitakerc blame it on the yellow c...


# Word Embeddings

In [44]:
# Create Tokens
token=[]
for text in sub_df["final_tweets"]:
  text = str(text)
  words=word_tokenize(text)
  token.extend(words)

In [45]:
token

['Drop',
 'it',
 'down',
 'for',
 'a',
 'niggah',
 'socass_',
 'Dude',
 'remember',
 'this',
 'wop',
 'video',
 'sexilexi',
 'http',
 't',
 'co',
 'P',
 'HEuBOQBu',
 'OMG',
 'WHERE',
 'DID',
 'YOU',
 'FIND',
 'THAT',
 'porn',
 'android',
 'iphone',
 'ipad',
 'sex',
 'xxx',
 'Masturbation',
 'A',
 'very',
 'hot',
 'black',
 'tranny',
 'is',
 'mastu',
 'http',
 't',
 'co',
 'OfGGMLXQAj',
 'RT',
 'tupactopus',
 'need',
 'a',
 'bad',
 'bitch',
 'who',
 'know',
 'how',
 'treat',
 'a',
 'pokemon',
 'master',
 'right',
 'RT',
 'Cam_Major',
 'naw',
 'no',
 'b',
 'tho',
 'if',
 'u',
 'under',
 'let',
 'say',
 'and',
 'u',
 'got',
 'like',
 'body',
 'u',
 'def',
 'a',
 'hoe',
 'lol',
 'there',
 'no',
 'debating',
 'that',
 'Let',
 'me',
 'text',
 'my',
 'hoe',
 'gm',
 'or',
 'sumpin',
 'RT',
 'sofancyy_',
 'A',
 'hoe',
 'gone',
 'be',
 'a',
 'hoe',
 'RT',
 'Thad_CastIe',
 'Regrets',
 'are',
 'for',
 'pussy',
 'Shit',
 'happens',
 'deal',
 'with',
 'it',
 'BITCH',
 'RT',
 'jla_b',
 'Stop',
 'thin

In [107]:
# Creating Vocabulary
from collections import Counter
vocab={'<unk>':0}
for token in Counter(token).keys():
  vocab[token]=len(vocab)

In [47]:
vocab

{'<unk': 0,
 'Drop': 1,
 'it': 2,
 'down': 3,
 'for': 4,
 'a': 5,
 'niggah': 6,
 'socass_': 7,
 'Dude': 8,
 'remember': 9,
 'this': 10,
 'wop': 11,
 'video': 12,
 'sexilexi': 13,
 'http': 14,
 't': 15,
 'co': 16,
 'P': 17,
 'HEuBOQBu': 18,
 'OMG': 19,
 'WHERE': 20,
 'DID': 21,
 'YOU': 22,
 'FIND': 23,
 'THAT': 24,
 'porn': 25,
 'android': 26,
 'iphone': 27,
 'ipad': 28,
 'sex': 29,
 'xxx': 30,
 'Masturbation': 31,
 'A': 32,
 'very': 33,
 'hot': 34,
 'black': 35,
 'tranny': 36,
 'is': 37,
 'mastu': 38,
 'OfGGMLXQAj': 39,
 'RT': 40,
 'tupactopus': 41,
 'need': 42,
 'bad': 43,
 'bitch': 44,
 'who': 45,
 'know': 46,
 'how': 47,
 'treat': 48,
 'pokemon': 49,
 'master': 50,
 'right': 51,
 'Cam_Major': 52,
 'naw': 53,
 'no': 54,
 'b': 55,
 'tho': 56,
 'if': 57,
 'u': 58,
 'under': 59,
 'let': 60,
 'say': 61,
 'and': 62,
 'got': 63,
 'like': 64,
 'body': 65,
 'def': 66,
 'hoe': 67,
 'lol': 68,
 'there': 69,
 'debating': 70,
 'that': 71,
 'Let': 72,
 'me': 73,
 'text': 74,
 'my': 75,
 'gm': 76,

In [49]:
# Tokenize Sentence

tokenize_sentence = [
    word_tokenize(str(sentence))
    for sentence in sub_df['final_tweets']
]

In [50]:
tokenize_sentence

[['Drop', 'it', 'down', 'for', 'a', 'niggah'],
 ['socass_',
  'Dude',
  'remember',
  'this',
  'wop',
  'video',
  'sexilexi',
  'http',
  't',
  'co',
  'P',
  'HEuBOQBu',
  'OMG',
  'WHERE',
  'DID',
  'YOU',
  'FIND',
  'THAT'],
 ['porn',
  'android',
  'iphone',
  'ipad',
  'sex',
  'xxx',
  'Masturbation',
  'A',
  'very',
  'hot',
  'black',
  'tranny',
  'is',
  'mastu',
  'http',
  't',
  'co',
  'OfGGMLXQAj'],
 ['RT',
  'tupactopus',
  'need',
  'a',
  'bad',
  'bitch',
  'who',
  'know',
  'how',
  'treat',
  'a',
  'pokemon',
  'master',
  'right'],
 ['RT',
  'Cam_Major',
  'naw',
  'no',
  'b',
  'tho',
  'if',
  'u',
  'under',
  'let',
  'say',
  'and',
  'u',
  'got',
  'like',
  'body',
  'u',
  'def',
  'a',
  'hoe',
  'lol',
  'there',
  'no',
  'debating',
  'that'],
 ['Let', 'me', 'text', 'my', 'hoe', 'gm', 'or', 'sumpin'],
 ['RT', 'sofancyy_', 'A', 'hoe', 'gone', 'be', 'a', 'hoe'],
 ['RT',
  'Thad_CastIe',
  'Regrets',
  'are',
  'for',
  'pussy',
  'Shit',
  'hap

In [52]:
def text_to_indices(sentence,vocab):
  numeric_sentence=[]
  for token in sentence:
    if token not in vocab:
      numeric_sentence.append(vocab['<unk>'])
    else:
      numeric_sentence.append(vocab[token])
  return numeric_sentence

In [54]:
input_numeric_sentence=[]
for sentence in tokenize_sentence:
  input_numeric_sentence.append(text_to_indices(sentence,vocab))

In [55]:
input_numeric_sentence

[[1, 2, 3, 4, 5, 6],
 [7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24],
 [25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 14, 15, 16, 39],
 [40, 41, 42, 5, 43, 44, 45, 46, 47, 48, 5, 49, 50, 51],
 [40,
  52,
  53,
  54,
  55,
  56,
  57,
  58,
  59,
  60,
  61,
  62,
  58,
  63,
  64,
  65,
  58,
  66,
  5,
  67,
  68,
  69,
  54,
  70,
  71],
 [72, 73, 74, 75, 67, 76, 77, 78],
 [40, 79, 32, 67, 80, 81, 5, 67],
 [40, 82, 83, 84, 4, 85, 86, 87, 88, 89, 2],
 [90, 40, 91, 92, 93, 94, 95, 96, 97, 98, 99, 44],
 [100,
  101,
  42,
  96,
  102,
  103,
  104,
  105,
  106,
  107,
  108,
  109,
  5,
  110,
  111,
  112,
  113,
  112,
  114,
  115,
  5,
  67,
  116,
  5,
  117],
 [118, 119, 120, 63, 121, 44, 4, 121, 122],
 [37, 123, 124, 125, 5, 126, 127, 44],
 [128,
  129,
  15,
  130,
  131,
  132,
  44,
  4,
  75,
  133,
  96,
  81,
  134,
  135,
  136,
  137,
  138,
  139,
  62,
  140,
  46,
  141,
  142,
  140,
  63,
  143,
  144],
 [145, 146, 147, 148, 121, 149

# Padding

In [56]:
len_list=[]
for sentence in input_numeric_sentence:
  len_list.append(len(sentence))

In [59]:
max_len=max(len_list)
max_len

33

In [60]:
padded_sequence=[]
for sequence in input_numeric_sentence:
  padded_sequence.append([0]*(max_len-len(sequence))+sequence)

In [61]:
padded_sequence

[[0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  1,
  2,
  3,
  4,
  5,
  6],
 [0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  7,
  8,
  9,
  10,
  11,
  12,
  13,
  14,
  15,
  16,
  17,
  18,
  19,
  20,
  21,
  22,
  23,
  24],
 [0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  25,
  26,
  27,
  28,
  29,
  30,
  31,
  32,
  33,
  34,
  35,
  36,
  37,
  38,
  14,
  15,
  16,
  39],
 [0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  40,
  41,
  42,
  5,
  43,
  44,
  45,
  46,
  47,
  48,
  5,
  49,
  50,
  51],
 [0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  40,
  52,
  53,
  54,
  55,
  56,
  57,
  58,
  59,
  60,
  61,
  62,
  58,
  63,
  64,
  65,
  58,
  66,
  5,
  67,
  68,
  69,
  54,
  70,
  71],
 [0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
 

In [67]:
padded_sequence=torch.tensor(padded_sequence,dtype=torch.long)

In [92]:
X=padded_sequence
y=torch.tensor(sub_df["class"].values, dtype=torch.long).to(device)

In [69]:
class CustomDataset(Dataset):
  def __init__(self,X,y):
    self.x=X
    self.y=y
  def __len__(self):
    return len(self.x)
  def __getitem__(self,indx):
    return self.x[indx],self.y[indx]

In [70]:
dataset=CustomDataset(X,y)

In [71]:
dataloader=DataLoader(dataset,batch_size=32,shuffle=True)

In [99]:
class LSTMmodel(nn.Module):
  def __init__(self,vocab_size, num_classes):
    super(LSTMmodel,self).__init__()
    self.embedding=nn.Embedding(vocab_size,100)
    self.lstm=nn.LSTM(100,150,batch_first=True)
    self.fc=nn.Linear(150, num_classes)
  def forward(self,x):
    embedded=self.embedding(x)
    intermidiate_hidden_state,(final_hidden_state,final_cell_state)=self.lstm(embedded)
    output=self.fc(final_hidden_state.squeeze(0))
    return output

In [100]:
model=LSTMmodel(len(vocab), 3)

In [101]:
model.to(device)

LSTMmodel(
  (embedding): Embedding(2869, 100)
  (lstm): LSTM(100, 150, batch_first=True)
  (fc): Linear(in_features=150, out_features=3, bias=True)
)

# Training Loop

In [103]:
epochs=30
criterion=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=0.001)

all_predictions = []
all_true_labels = []

for epoch in range(epochs):
  total_loss=0
  for batch_x,batch_y in dataloader:
    batch_x,batch_y=batch_x.to(device),batch_y.to(device)
    optimizer.zero_grad()
    output=model(batch_x)
    loss=criterion(output.squeeze(0),batch_y)
    loss.backward()
    optimizer.step()
    total_loss+=loss.item()
    all_predictions.extend(torch.argmax(output.squeeze(0), dim=1).cpu().numpy())
    all_true_labels.extend(batch_y.cpu().numpy())

  print(f"Epoch: {epoch + 1}, Loss: {total_loss:.4f}")

Epoch: 1, Loss: 15.8575
Epoch: 2, Loss: 10.2799
Epoch: 3, Loss: 9.6793
Epoch: 4, Loss: 9.0488
Epoch: 5, Loss: 7.7181
Epoch: 6, Loss: 6.1865
Epoch: 7, Loss: 4.3114
Epoch: 8, Loss: 4.0407
Epoch: 9, Loss: 2.8718
Epoch: 10, Loss: 1.5866
Epoch: 11, Loss: 0.9685
Epoch: 12, Loss: 0.6190
Epoch: 13, Loss: 0.4047
Epoch: 14, Loss: 0.2786
Epoch: 15, Loss: 0.1877
Epoch: 16, Loss: 0.1340
Epoch: 17, Loss: 0.1027
Epoch: 18, Loss: 0.0806
Epoch: 19, Loss: 0.0633
Epoch: 20, Loss: 0.0520
Epoch: 21, Loss: 0.0432
Epoch: 22, Loss: 0.0364
Epoch: 23, Loss: 0.0317
Epoch: 24, Loss: 0.0276
Epoch: 25, Loss: 0.0237
Epoch: 26, Loss: 0.0209
Epoch: 27, Loss: 0.0189
Epoch: 28, Loss: 0.0169
Epoch: 29, Loss: 0.0149
Epoch: 30, Loss: 0.0136


In [102]:
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(all_true_labels, all_predictions)
print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.9566


# Prediction Function

In [111]:
def prediction(model,vocab,text):
  lemmatize_text=lemmatization(text)
  stop_words_text=stop_words(lemmatize_text)
  token=word_tokenize(stop_words_text)
  neumeric_text=text_to_indices(token,vocab)
  padded_text=torch.tensor([0]*(max_len-len(neumeric_text))+neumeric_text,dtype=torch.long).unsqueeze(0).to(device)
  output=model(padded_text)
  print(torch.argmax(output.squeeze(0), dim=0).item())

In [112]:
prediction(model,vocab,"Murda Gang Bitch its Gang Land")

2
